# ENCS5342 Final Project - Track 1
        ## Part-of-Speech (POS) Tagging

        **Team members**

        | Name | Student ID |
        |---|---|
        | Aysha Hasan | 1220352 |
        | Besan Maaly | 1222776 |

        **Course:** ENCS5342 - Information Retrieval with Applications of NLP  
        **Dataset:** English Penn Treebank in CoNLL format  
        **Task:** Predict joint POS tags by combining coarse and fine POS tags, such as `NOUN-NNP` and `VERB-VBD`.

## 1. Abstract

        Write a concise summary of the project here.

        Include:

        - The POS tagging problem.
        - The dataset used.
        - The two compared approaches, for example HMM baseline and BiLSTM.
        - The main training and development results after you run the experiments.
        - One short conclusion about which model worked better and why.

## 2. Introduction

        Explain the motivation for POS tagging and why it is useful in NLP pipelines.

        Suggested points:

        - POS tagging assigns a grammatical category to every token in a sentence.
        - The same word may have different POS tags depending on context.
        - POS tags support downstream NLP tasks such as parsing, information extraction, search, and machine translation.
        - In this project, the goal is to build POS taggers from scratch and compare a probabilistic baseline with a neural sequence model.

        **Contributions**

        - Parsed the CoNLL dataset and constructed joint coarse+fine POS labels.
        - Implemented a Hidden Markov Model baseline with Viterbi decoding.
        - Implemented a BiLSTM neural tagger.
        - Evaluated both models using token accuracy, per-class precision/recall/F1, micro averages, and macro averages.

## 3. Prior & Related Work

        Add at least one related work citation and briefly explain how it connects to your project.

        Suggested related work:

        - Toutanova et al. (2003), "Feature-Rich Part-of-Speech Tagging with a Cyclic Dependency Network".
        - Huang, Xu, and Yu (2015), "Bidirectional LSTM-CRF Models for Sequence Tagging".

        Write in your own words:

        - What the paper/model does.
        - Why it is relevant to POS tagging.
        - How your project is simpler or different.

## 4. Model

        This section should describe the models used in the project.

### 4.1 Hidden Markov Model Baseline

The first model is a Hidden Markov Model (HMM) baseline. It models POS tagging as a sequence labeling problem where each hidden state is a joint POS tag and each observed symbol is a word token.

For a sentence $w_1, \dots, w_n$ and tag sequence $t_1, \dots, t_n$, the HMM uses three probability tables estimated from the training set:

- Initial probabilities: $\pi(t) = P(t_1=t)$
- Transition probabilities: $A(t_i \mid t_{i-1})$
- Emission probabilities: $B(w_i \mid t_i)$

The implementation applies additive smoothing to transition and emission probabilities. Rare training words are mapped to feature-based unknown classes such as UNK-CAP, UNK-ING, UNK-NUM, and UNK-HYPH, which gives the model a better fallback for unseen development words. Decoding is performed with the Viterbi algorithm in log space to avoid numerical underflow.


In [ ]:
from data_loader import load_conll, get_vocab_and_tag_sets
from vocab import build_word2idx, build_tag2idx
from hmm_baseline import HMMTagger, explain_sentence_log
from viterbi import viterbi, predict


### 4.2 BiLSTM Neural Tagger

        Describe the neural model:

        - Tokens are mapped to word indices.
        - An embedding layer converts word indices to dense vectors.
        - A bidirectional LSTM reads the sentence in both directions.
        - A linear layer predicts one POS tag per token.
        - Padding tokens are ignored in the loss and evaluation.

        Mention ML engineering techniques you used, such as dropout, early stopping, gradient clipping, and weight decay.

In [ ]:
# TODO: Import and/or paste your BiLSTM model here.
        # Suggested option:
        # from src.bilstm_model import BiLSTMTagger
        # from src.train_utils import POSDataset, collate_fn

## 5. Data

        Describe the dataset:

        - English Penn Treebank in CoNLL format.
        - Training file: `data/en-universal-train.conll`.
        - Development file: `data/en-universal-dev.conll`.
        - The held-out test set is not available to students.
        - The target label is a joint tag: `CPOS-FPOS`.

        Explain preprocessing:

        - Read each sentence as a list of tokens.
        - Use column 2 as word form.
        - Use columns 4 and 5 to build the joint POS tag.
        - Build vocabularies for words and tags.
        - Pad variable-length sentences for neural batching.

In [ ]:
# Basic setup
from pathlib import Path
import random
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"

import sys
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)


In [ ]:
train_sentences = load_conll(DATA_DIR / "en-universal-train.conll")
dev_sentences = load_conll(DATA_DIR / "en-universal-dev.conll")

train_vocab, train_tags = get_vocab_and_tag_sets(train_sentences)
word2idx = build_word2idx(train_vocab)
tag2idx = build_tag2idx(train_tags)

print("Training sentences:", len(train_sentences))
print("Development sentences:", len(dev_sentences))
print("Vocabulary size including <PAD>/<UNK>:", len(word2idx))
print("Number of joint POS tags including <PAD>:", len(tag2idx))

words, tags = train_sentences[0]
print("\nFirst training sentence:")
for word, tag in zip(words, tags):
    print(f"{word:15s} -> {tag}")


In [ ]:
from collections import Counter

def count_tokens(sentences):
    return sum(len(words) for words, _ in sentences)

def sentence_length_summary(sentences):
    lengths = np.array([len(words) for words, _ in sentences])
    return {
        "min": int(lengths.min()),
        "mean": float(lengths.mean()),
        "median": float(np.median(lengths)),
        "max": int(lengths.max()),
    }

tag_counter = Counter(tag for _, tags in train_sentences for tag in tags)

print("Train tokens:", count_tokens(train_sentences))
print("Dev tokens:", count_tokens(dev_sentences))
print("Unique train words:", len(train_vocab))
print("Unique joint POS tags:", len(train_tags))
print("Train sentence length summary:", sentence_length_summary(train_sentences))
print("Dev sentence length summary:", sentence_length_summary(dev_sentences))

print("\nMost frequent training tags:")
for tag, count in tag_counter.most_common(10):
    print(f"{tag:12s} {count}")


## 6. Experiments

        Describe the experimental setup.

        Include:

        - Data splits used for training and development evaluation.
        - Hyperparameters for HMM smoothing and unknown-word handling.
        - Hyperparameters for BiLSTM: embedding dimension, hidden dimension, batch size, learning rate, epochs, dropout, optimizer.
        - Evaluation metrics: per-class precision/recall/F1, micro average, macro average, and token accuracy.
        - Any early stopping, gradient clipping, or regularization settings.

In [ ]:
from evaluation import evaluate, print_report


### 6.1 HMM Experiment

In [ ]:
hmm = HMMTagger(k_emission=0.1, k_transition=0.1, rare_threshold=1)
hmm.fit(train_sentences)

print("HMM model statistics")
print("Number of tags:", hmm.num_tags)
print("Vocabulary size after rare-word mapping:", hmm.vocab_size)
print("Rare words mapped to UNK classes:", len(hmm.rare_words))

print("\nSample probability checks")
for tag in ["DET-DT", "NOUN-NNP", "VERB-VBD", "ADP-IN"]:
    if tag in hmm.tags:
        print(f"pi({tag}) = {hmm.initial_prob(tag):.6f}")

print(f"A(NOUN-NN | DET-DT) = {hmm.transition_prob('DET-DT', 'NOUN-NN'):.6f}")
print(f"B('the' | DET-DT) = {hmm.emission_prob('DET-DT', 'the'):.6f}")
print(f"B('policy-making' | NOUN-NN) = {hmm.emission_prob('NOUN-NN', 'policy-making'):.6f}")

sample_words, sample_gold = dev_sentences[0]
sample_pred, sample_score = viterbi(hmm, sample_words)
print("\nViterbi demo on first development sentence")
print("Sentence:", " ".join(sample_words))
print(f"Predicted sequence log-score: {sample_score:.4f}\n")
print(f"{'Word':15s} {'Gold':15s} {'Predicted':15s} Match")
print("-" * 55)
for word, gold, pred in zip(sample_words, sample_gold, sample_pred):
    print(f"{word:15s} {gold:15s} {pred:15s} {gold == pred}")

print("\nRunning Viterbi on training set...")
train_hmm_preds = predict(hmm, train_sentences, progress_every=1000)
train_hmm_results = evaluate(train_sentences, train_hmm_preds)

print("\nRunning Viterbi on development set...")
dev_hmm_preds = predict(hmm, dev_sentences, progress_every=1000)
dev_hmm_results = evaluate(dev_sentences, dev_hmm_preds)

print_report(train_hmm_results, "HMM Baseline - Training Set")
print_report(dev_hmm_results, "HMM Baseline - Development Set")


### 6.2 BiLSTM Experiment

In [ ]:
# TODO: Build vocabularies, datasets, dataloaders, optimizer, loss function, and training loop.
        #
        # Important:
        # - use ignore_index=0 in CrossEntropyLoss to ignore padded tags
        # - use gradient clipping
        # - track training/dev accuracy or F1 after each epoch
        # - save the best model based on development score

In [ ]:
# TODO: Evaluate the BiLSTM on train and dev sets.
        #
        # train_bilstm_results = ...
        # dev_bilstm_results = ...
        #
        # print_report(train_bilstm_results, "BiLSTM - Training Set")
        # print_report(dev_bilstm_results, "BiLSTM - Development Set")

## 7. Results

        Present quantitative results in clear tables.

        Required:

        - Training results for every model.
        - Development results for every model.
        - Per-class precision, recall, and F1.
        - Micro-averaged precision, recall, and F1.
        - Macro-averaged precision, recall, and F1.
        - Overall token-level accuracy.

        Strongly encouraged:

        - Confusion matrix.
        - Learning curves for BiLSTM.
        - A compact comparison table with HMM vs BiLSTM.

In [ ]:
def metrics_row(model, split, results):
    return {
        "Model": model,
        "Split": split,
        "Accuracy": results["accuracy"],
        "Micro Precision": results["micro"]["precision"],
        "Micro Recall": results["micro"]["recall"],
        "Micro F1": results["micro"]["f1"],
        "Macro Precision": results["macro"]["precision"],
        "Macro Recall": results["macro"]["recall"],
        "Macro F1": results["macro"]["f1"],
    }

comparison_rows = []
if "train_hmm_results" in globals() and "dev_hmm_results" in globals():
    comparison_rows.extend([
        metrics_row("HMM", "Train", train_hmm_results),
        metrics_row("HMM", "Dev", dev_hmm_results),
    ])

if "train_bilstm_results" in globals() and "dev_bilstm_results" in globals():
    comparison_rows.extend([
        metrics_row("BiLSTM", "Train", train_bilstm_results),
        metrics_row("BiLSTM", "Dev", dev_bilstm_results),
    ])

try:
    import pandas as pd
    results_table = pd.DataFrame(comparison_rows)
    display(results_table.style.format({
        "Accuracy": "{:.4f}",
        "Micro Precision": "{:.4f}",
        "Micro Recall": "{:.4f}",
        "Micro F1": "{:.4f}",
        "Macro Precision": "{:.4f}",
        "Macro Recall": "{:.4f}",
        "Macro F1": "{:.4f}",
    }))
except ImportError:
    for row in comparison_rows:
        print(row)


In [ ]:
# TODO: Plot learning curves or confusion matrix if available.

## 8. Analysis & Conclusion

        Discuss the results in your own words.

        Suggested analysis questions:

        - Which model performed better on the development set?
        - Is there a gap between training and development performance?
        - Which POS tags were easiest?
        - Which POS tags were hardest?
        - Is macro F1 lower than micro F1? If yes, explain how rare classes affect this.
        - What kinds of errors happen often?
        - What are the limitations of your approach?

        Conclude with a short summary of what you learned and what could be improved in future work.

In [ ]:
# TODO: Optional error analysis.
        # Print examples where the model made mistakes.

## 9. AI Usage Statement

        Write the required AI usage disclosure here.

        Example wording to edit:

        > We used AI tools for debugging, organization suggestions, and feedback on notebook structure. The core modeling decisions, experiment execution, result verification, and final analysis were reviewed and completed by the team. We are responsible for every claim, result, and line of code in this submission.

        Also mention which parts were written entirely by the team without AI assistance.

## 10. References

        Add all references used in the project.

        Suggested references:

        - Toutanova, K., Klein, D., Manning, C. D., & Singer, Y. (2003). Feature-Rich Part-of-Speech Tagging with a Cyclic Dependency Network. HLT-NAACL.
        - Huang, Z., Xu, W., & Yu, K. (2015). Bidirectional LSTM-CRF Models for Sequence Tagging. arXiv:1508.01991.
        - PyTorch documentation.
        - English Penn Treebank / CoNLL dataset source provided by the course instructor.